<a href="https://colab.research.google.com/github/SurajTamang-22/CodeAlpha_MUSIC-PLAYER/blob/main/Text_to_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update
!apt-get install -y zstd
!apt-get install -y lspci

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,855 kB]
Get:13 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,930 kB]
Get:14 http://

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Install python libraries
!pip install requests sqlite-utils

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.5/68.5 kB 2.7 MB/s eta 0:00:00


In [ ]:
# Start Ollama in background
!nohup ollama serve > ollama.log 2>&1 &

In [ ]:
!ollama pull llama3


In [ ]:
import sqlite3

# Create database
conn = sqlite3.connect("company.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE employees (
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INTEGER,
    age INTEGER
)
""")

data = [
(1,"Rahul","IT",60000,30),
(2,"Anita","HR",45000,28),
(3,"Vikram","Finance",70000,35),
(4,"Sneha","IT",55000,27),
(5,"Arjun","Marketing",50000,32)
]

cursor.executemany("INSERT INTO employees VALUES (?,?,?,?,?)",data)

conn.commit()
conn.close()

print("Database created successfully")

Database created successfully


In [ ]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"

def text_to_sql(question):

    prompt = f"""
You are an expert SQL generator.

Database Schema:
Table: employees
Columns:
id, name, department, salary, age

Convert the following question into a SQL query.

Question: {question}

Only return SQL query.
"""

    payload = {
        "model": "llama3",
        "prompt": prompt,
        "stream": False
    }

    response = requests.post(OLLAMA_URL, json=payload)
    result = response.json()

    return result["response"].strip()

In [ ]:
import sqlite3

def run_sql(query):

    conn = sqlite3.connect("company.db")
    cursor = conn.cursor()

    try:
        cursor.execute(query)
        rows = cursor.fetchall()
        conn.close()
        return rows
    except Exception as e:
        conn.close()
        return str(e)

In [ ]:
def ask_database(question):

    sql_query = text_to_sql(question)

    print("Generated SQL:")
    print(sql_query)

    result = run_sql(sql_query)

    print("\nQuery Result:")
    print(result)

In [ ]:
import gradio as gr

def chat_sql(question):
    sql = text_to_sql(question)
    result = run_sql(sql)
    return f"SQL Query:\n{sql}\n\nResult:\n{result}"

gr.Interface(
    fn=chat_sql,
    inputs="text",
    outputs="text"
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://80c9cf65d3c76dfd9b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
